In [1]:
import pandas as pd
import numpy as np
print(np.__version__)
print(pd.__version__)

1.26.4
2.3.3


In [ ]:
# Load CSV files
orders = pd.read_csv('E:\Dissertation Sem\Papers for dissertation\orders.csv')
order_prior = pd.read_csv('E:\Dissertation Sem\Papers for dissertation\order_products__prior.csv')
order_train = pd.read_csv('E:\Dissertation Sem\Papers for dissertation\order_products__train.csv')
products = pd.read_csv('E:\Dissertation Sem\Papers for dissertation\products.csv')
aisles = pd.read_csv('E:\\Dissertation Sem\\Papers for dissertation\\aisles.csv')
departments = pd.read_csv('E:\Dissertation Sem\Papers for dissertation\departments.csv')

# Merge prior and train order-product data (all purchases)
order_products = pd.concat([order_prior, order_train], ignore_index=True)
# Merge product details
order_products = order_products.merge(products, on='product_id', how='left')
order_products = order_products.merge(aisles, on='aisle_id', how='left')
order_products = order_products.merge(departments, on='department_id', how='left')
# Merge user and order info
order_products = order_products.merge(orders[['order_id','user_id','order_number','order_dow',
                                             'order_hour_of_day','days_since_prior_order']],
                                      on='order_id', how='left')
print(order_products.head(5))

In [ ]:
# Filter out users with fewer than 5 orders
user_order_counts = orders.groupby('user_id')['order_id'].count()
active_users = user_order_counts[user_order_counts >= 5].index
orders = orders[orders['user_id'].isin(active_users)]
order_products = order_products[order_products['user_id'].isin(active_users)]

# Filter out very rare products (e.g., products bought < 10 times in total)
product_counts = order_products['product_id'].value_counts()
popular_products = product_counts[product_counts >= 10].index
order_products = order_products[order_products['product_id'].isin(popular_products)]

# Update the dataframes accordingly
print("Users after filtering:", order_products['user_id'].nunique())
print("Products after filtering:", order_products['product_id'].nunique())


In [ ]:
# Sort order_products by user, then by order_number and add_to_cart_order
order_products = order_products.sort_values(['user_id','order_number','add_to_cart_order'])
# Aggregate product_id sequence per user
user_sequences = order_products.groupby('user_id')['product_id'].apply(list)
print(user_sequences.head(3))

In [ ]:
# Filter only completed orders
completed_orders = order_products.dropna(subset=['user_id', 'order_number', 'days_since_prior_order'])

# RFM components
rfm = completed_orders.groupby('user_id').agg({
    'order_number': 'max',  # Frequency (number of orders)
    'days_since_prior_order': 'mean',  # Avg delay between orders
    'product_id': 'count'  # Basket size (proxy for Monetary)
}).reset_index()

# Rename columns
rfm.columns = ['user_id', 'frequency', 'avg_days_between_orders', 'avg_basket_size']

# Optional: log transform frequency to reduce skew
rfm['log_frequency'] = np.log1p(rfm['frequency'])

# Drop outliers (optional)
rfm = rfm[(rfm['avg_days_between_orders'] < 30) & (rfm['avg_basket_size'] < 100)]

rfm.head()